In [1]:
from ultrack import MainConfig, Tracker
from tracktour import load_tiff_frames

from pathlib import Path
import os

ds_dir = '/home/ddon0001/PhD/data/cell_tracking_challenge/SUBMISSION'
res_root_dir = '/home/ddon0001/PhD/experiments/ultrack/'

datasets = [ds for ds in os.listdir(ds_dir) if os.path.isdir(os.path.join(ds_dir, ds)) and ds != 'SW']

err_seg_paths = [(ds, seq) for ds in datasets for seq in os.listdir(os.path.join(ds_dir, ds)) if 'ERR_SEG' in seq]

In [ ]:
errors = {}
for ds, seq in err_seg_paths:
    label_path = os.path.join(ds_dir, ds, seq)
    out_path = os.path.join(res_root_dir, f"{ds}_{seq.replace('ERR_SEG', 'RES')}")

    if os.path.exists(out_path):
        print(f"Skipping {ds} {seq} as output path exists.")
        continue
    print(f"Processing {ds} {seq}...")

    labels = load_tiff_frames(label_path)
    
    # Create a config
    config = MainConfig()

    # this removes irrelevant segments from the image
    # see the configuration section for more details
    config.segmentation_config.min_frontier = 0.5

    # Run the tracking
    tracker = Tracker(config=config)
    try:
        tracker.track(labels=labels, overwrite='all')
        tracker.to_ctc(Path(out_path))
    except Exception as e:
        print(f"Error processing {ds} {seq}: {e}")
        errors[(ds, seq)] = str(e)
        continue


We first run ultrack with its suggested config from the docs. Everything is kept to default parameters, except `segmentation_config.min_frontier=0.5`. Six datasets fail to solve, as listed below.

```
{('Fluo-C2DL-MSC',
  '02_ERR_SEG'): 'No links found for time 0. Increase `linking_config.max_distance` parameter.',
 ('BF-C2DL-MuSC',
  '02_ERR_SEG'): 'No links found for time 188. Increase `linking_config.max_distance` parameter.',
 ('BF-C2DL-MuSC',
  '01_ERR_SEG'): 'No links found for time 221. Increase `linking_config.max_distance` parameter.',
 ('BF-C2DL-HSC',
  '01_ERR_SEG'): 'No links found for time 104. Increase `linking_config.max_distance` parameter.',
 ('Fluo-N3DH-CE',
  '02_ERR_SEG'): 'No links found for time 3. Increase `linking_config.max_distance` parameter.',
 ('Fluo-N3DH-CE',
  '01_ERR_SEG'): 'No links found for time 4. Increase `linking_config.max_distance` parameter.'}
```

When we look at ctc evaluation, we see there's not a lot of FP nodes (splitting segments), but we've introduced a lot of new FN and NS nodes. This implies we're likely merging segments too zealously. We try running ultrack again, this time relaxing any parameters that may lead to improper merging of segments, guided by [the docs](https://royerlab.github.io/ultrack/optimizing.html#tracking-tuning). Specifically:

```python
segmentation_config.min_frontier = 0.0
segmentation_config.min_area = 20
linking_config.max_distance = 50
linking_config.max_neighbors = 10
```

This seems to improve results, but we get more datasets erroring, and are advised in the code to increase `min_area` to avoid this. We therefore re-run errored datasets with `min_area=50`.

```
{('Fluo-C3DL-MDA231', '01_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('Fluo-C2DL-MSC', '02_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('Fluo-C3DH-H157', '02_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('Fluo-N3DH-CHO', '02_ERR_SEG'): 'Region too small. Size of 5 found.',
 ('BF-C2DL-MuSC', '01_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('BF-C2DL-HSC', '02_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('Fluo-N2DH-SIM+', '02_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('Fluo-N2DH-SIM+', '01_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('PhC-C2DL-PSC', '02_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('PhC-C2DL-PSC', '01_ERR_SEG'): 'Region too small. Size of 6 found.',
 ('Fluo-N3DH-CE', '01_ERR_SEG'): 'Region too small. Size of 5 found.'}
```

New config:

```python
segmentation_config.min_frontier = 0.0
segmentation_config.min_area = 50
linking_config.max_distance = 50
linking_config.max_neighbors = 10
```

Finally, we are left with two errors:

```
{('Fluo-C2DL-MSC',
  '02_ERR_SEG'): 'No links found for time 3. Increase `linking_config.max_distance` parameter.',
 ('BF-C2DL-MuSC',
  '01_ERR_SEG'): 'No links found for time 649. Increase `linking_config.max_distance` parameter.'}
```

To address these, we update `max_distance=100` for a new config:

```python
segmentation_config.min_frontier = 0.0
segmentation_config.min_area = 50
linking_config.max_distance = 100
linking_config.max_neighbors = 10
```

To see how ultrack behaves by default, we also do a run with no change in settings whatsoever, and we get results for most datasets, with the following errors:

```
{('Fluo-C2DL-MSC',
  '02_ERR_SEG'): 'No links found for time 0. Increase `linking_config.max_distance` parameter.',
 ('BF-C2DL-MuSC',
  '02_ERR_SEG'): 'No links found for time 188. Increase `linking_config.max_distance` parameter.',
 ('BF-C2DL-MuSC',
  '01_ERR_SEG'): 'No links found for time 221. Increase `linking_config.max_distance` parameter.',
 ('BF-C2DL-HSC',
  '01_ERR_SEG'): 'No links found for time 104. Increase `linking_config.max_distance` parameter.',
 ('Fluo-N3DH-CE',
  '02_ERR_SEG'): 'No links found for time 3. Increase `linking_config.max_distance` parameter.',
 ('Fluo-N3DH-CE',
  '01_ERR_SEG'): 'No links found for time 4. Increase `linking_config.max_distance` parameter.'}
```

We run the remaining datasets with an increased value of `max_distance=100`.

In [ ]:
errors = {}
for ds, seq in err_seg_paths:
    label_path = os.path.join(ds_dir, ds, seq)
    out_path = os.path.join(res_root_dir, f"{ds}_{seq.replace('ERR_SEG', 'default_RES')}")

    if os.path.exists(out_path):
        print(f"Skipping {ds} {seq} as output path exists.")
        continue
    print(f"Processing {ds} {seq}...")

    labels = load_tiff_frames(label_path)
    
    # Create a config
    config = MainConfig()

    # config.segmentation_config.min_frontier = 0.0
    # config.segmentation_config.min_area = 50

    config.linking_config.max_distance = 100
    # config.linking_config.max_neighbors = 10

    # Run the tracking
    tracker = Tracker(config=config)
    try:
        tracker.track(labels=labels, overwrite='all')
        tracker.to_ctc(Path(out_path))
    except Exception as e:
        print(f"Error processing {ds} {seq}: {e}")
        errors[(ds, seq)] = str(e)
        continue

In [11]:
errors

{}